# Train an axon/dendrite classifier on proofread cells

Builds training data from the proofread cells the `cortical_cells` skeleton stage (`s1_skeletons`) has already written to the bucket, and trains a per-vertex classifier on the distance-window feature set.

1. **Cells**: roots with both axon and dendrite proofread, read straight from the bucket. Nothing is fetched from CAVE except the proofreading table.
2. **Labels and QC**: labels are the service's synapse-flow compartments. A cell contributes only if few of its outputs sit on dendrite and few of its inputs sit on axon.
3. **Features**: `DISTANCE_FEATURE_SPEC` over downstream windows of 10, 30 and 100 µm. Each cell is also featurized with a random share of its synapses removed, keeping the labels from the full set, so the model sees sparse cells with known answers.
4. **Train** with cells, not vertices, held out.
5. **Evaluate** against the bundled model on the same held-out cells, at full and thinned synapse density.
6. **Save** the model.

In [1]:
import os
from pathlib import Path

DATASET = "minnie65"   # or "v1dd"

# Where s1_skeletons wrote the cells, and the materialization whose root ids they are named by.
# cell_types is a view or table with a cell_type column; sparse_type is the type with the sparsest dendritic inputs.
SOURCES = {
    "minnie65": dict(datastack="minnie65_public", version=1822, cells="gs://csm-skel/osy/minnie65/skel", bundled_model="minnie65_ds15_us0_bd0.json", cell_types=dict(view="aibs_cell_info"), sparse_type="5P-NP"),
    "v1dd": dict(datastack="v1dd", version=1643, cells="gs://csm-skel/osy/v1dd/skel", bundled_model="v1dd_ds15_us0_bd0.json", cell_types=dict(table="cell_type_multifeature_v1"), sparse_type="L5NP"),
}
SOURCE = SOURCES[DATASET]

# Feature tables and the trained model are written here; cells are read from the bucket each time.
CACHE_DIR = Path.home() / ".cache" / "ossify_compartment_training" / f"{DATASET}_v{SOURCE['version']}"

MAX_CELLS = int(os.environ.get("MAX_CELLS", 0)) or None  # None = every proofread cell in the bucket
N_WORKERS = 16                # cells read from the bucket at once
SEED = 0

# A cell contributes to training only if it passes these.
QC = dict(max_pre_off_fraction=0.1, max_post_off_fraction=0.1, min_synapses=100)

# Share of synapses kept when thinning. Every cell is featurized at 1.0 and at one of the others.
THIN_LEVELS = [0.5, 0.2, 0.1, 0.05]
# Cell types only balance the training data. They are never a feature, so none are needed to predict.
BOOST_TYPES = [SOURCE["sparse_type"]]  # cells of these types are featurized at every thinning level, not one
ROWS_PER_CELL = 3000          # vertices sampled per cell per pass, to keep the table manageable
HOLDOUT_FRACTION = 0.2        # of cells

# The tree decode, as s1_skeletons runs it. Saved with the model.
COSTS = {("axon", "dendrite"): 50, ("dendrite", "axon"): 2}
ABSORB_MIN_SIZE = 10

In [2]:
import json
import logging
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import xgboost
from caveclient import CAVEclient
from cloudfiles import CloudFiles
from sklearn.model_selection import GroupShuffleSplit

import ossify
from ossify import algorithms
from ossify.base import Link
from ossify.compartments import (
    DISTANCE_FEATURE_SPEC,
    DISTANCE_WINDOWS,
    ProbaVertexModel,
    StructuredLabeler,
    make_skel_prop_df,
)
from ossify.structured_prediction import TransitionSchema
from ossify.translate import SWCCompartment

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)

FEATURE_DIR = CACHE_DIR / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

cell_files = ossify.CellFiles(SOURCE["cells"])

def load(root_id):
    return cell_files.load(f"{root_id}.osy")

client = CAVEclient(SOURCE["datastack"])
client.materialize.version = SOURCE["version"]
print(DATASET, SOURCE["datastack"], "v", SOURCE["version"], "cells from", SOURCE["cells"])

minnie65 minnie65_public v 1822 cells from gs://csm-skel/osy/minnie65/skel


## 1. Cells

In [3]:
# Queried at the version's timestamp, which still works after the version itself has expired.
proof = client.materialize.live_query(
    "proofreading_status_and_strategy", timestamp=client.materialize.get_timestamp(SOURCE["version"])
)
proof = proof[proof.status_axon & proof.status_dendrite].drop_duplicates("pt_root_id")
proofread = proof.pt_root_id.astype("int64").to_numpy()

in_bucket = CloudFiles(SOURCE["cells"]).exists([f"{r}.osy" for r in proofread])
root_ids = np.array([r for r in proofread if in_bucket[f"{r}.osy"]])
print(len(proofread), "cells with axon and dendrite proofread;", len(root_ids), "of them in the bucket")
if MAX_CELLS:
    root_ids = np.random.default_rng(SEED).choice(root_ids, MAX_CELLS, replace=False)

# One cell type per root, from the view or table named in SOURCES.
if "view" in SOURCE["cell_types"]:
    info = client.materialize.query_view(SOURCE["cell_types"]["view"])
else:
    info = client.materialize.live_query(
        SOURCE["cell_types"]["table"], timestamp=client.materialize.get_timestamp(SOURCE["version"])
    )
cell_type = info.drop_duplicates("pt_root_id").set_index("pt_root_id").cell_type

def type_of(root_ids):
    return cell_type.reindex(np.asarray(root_ids)).fillna("unknown").to_numpy()

2220 cells with axon and dendrite proofread; 2219 of them in the bucket


## 2. Labels and QC

Labels are the `compartment` feature the skeleton service provides, which it makes by synapse flow. (The cells also carry `is_axon`, the bundled model's own prediction; it is not used as a label.) The QC asks how cleanly that split separates outputs from inputs.

In [4]:
def is_axon_of(cell):
    return np.asarray(cell.skeleton.get_feature("compartment")) == int(SWCCompartment.AXON)

def qc_row(root_id):
    cell = load(root_id)
    is_axon = is_axon_of(cell)
    row = algorithms.synapse_compartment_qc(cell, is_axon, **QC)
    compartment = cell.skeleton.get_feature("compartment")
    row.update(
        root_id=int(root_id),
        has_soma=bool(compartment[cell.skeleton.root_positional] == int(SWCCompartment.SOMA)),
        n_vertices=cell.skeleton.n_vertices,
        axon_fraction=float(is_axon.mean()),
    )
    return row

with ThreadPoolExecutor(N_WORKERS) as pool:
    rows = list(pool.map(qc_row, root_ids))
qc = pd.DataFrame(rows).set_index("root_id")
# A training cell also needs a soma to root at and some of both compartments.
qc["use"] = qc.passes & qc.has_soma & qc.axon_fraction.between(0.02, 0.98)
print(int(qc.use.sum()), "of", len(qc), "cells pass")
qc[["n_pre", "n_post", "pre_off_fraction", "post_off_fraction", "segregation_index", "axon_fraction", "has_soma", "use"]].describe(include="all").T

1969 of 2219 cells pass


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
n_pre,2219.0,NaN,NaN,NaN,1097.271293,1744.452473,11.0,311.5,555.0,978.0,15848.0
n_post,2219.0,NaN,NaN,NaN,3807.525011,2447.844228,394.0,2236.5,3005.0,4665.5,26797.0
pre_off_fraction,2219.0,NaN,NaN,NaN,0.057779,0.144273,0.001089,0.010662,0.021413,0.04524,1.0
post_off_fraction,2219.0,NaN,NaN,NaN,0.005346,0.006532,0.0,0.002711,0.004419,0.006405,0.094627
segregation_index,2219.0,NaN,NaN,NaN,0.866804,0.143292,0.0,0.860549,0.903956,0.92928,0.983167
axon_fraction,2219.0,NaN,NaN,NaN,0.632982,0.168196,0.0,0.553156,0.664832,0.750335,0.903994
has_soma,2219,1,True,2219,NaN,NaN,NaN,NaN,NaN,NaN,NaN
use,2219,2,True,1969,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Features

Windows are downstream only, so a branch is judged by what it leads to. Thinning removes synapses at random before the features are computed; the label stays what the full synapse set gave.

In [5]:
def thin_synapses(cell, keep_fraction, rng):
    """A copy of the cell with a random share of its synapses kept."""
    if keep_fraction >= 1:
        return cell
    thinned = cell.copy()
    for side in ("pre", "post"):
        name = f"{side}_syn"
        synapses = cell.annotations[name]
        kept = synapses.nodes[rng.random(len(synapses)) < keep_fraction].copy()
        thinned.remove_annotation(name)
        thinned.add_point_annotations(
            name,
            vertices=kept,
            spatial_columns=synapses.spatial_columns,
            linkage=Link(mapping=f"{side}_pt_l2_id", target="graph"),
        )
    return thinned

def features_for(cell, keep_fraction, rng):
    """Feature rows for one cell at one synapse density, with labels from the full cell."""
    skel = cell.skeleton
    df = make_skel_prop_df(
        thin_synapses(cell, keep_fraction, rng),
        downstream_hops=0,
        feature_spec=DISTANCE_FEATURE_SPEC,
        windows=DISTANCE_WINDOWS,
    ).reset_index(drop=True)
    df["is_axon"] = is_axon_of(cell)
    df["cable_um"] = np.asarray(skel.half_edge_length) / 1000
    df["vertex"] = np.arange(skel.n_vertices)
    df["keep_fraction"] = keep_fraction
    return df.drop(index=skel.root_positional)  # the soma is neither

def cell_features(root_id):
    """Full-density and thinned passes for a cell, cached as parquet.

    Most cells get one thinned pass at a random level; BOOST_TYPES cells get every level.
    """
    boosted = type_of([root_id])[0] in BOOST_TYPES
    path = FEATURE_DIR / (f"{root_id}_all_levels.parquet" if boosted else f"{root_id}.parquet")
    if path.exists():
        return pd.read_parquet(path)
    rng = np.random.default_rng([SEED, int(root_id) % 2**32])
    cell = load(root_id)
    levels = THIN_LEVELS if boosted else [float(rng.choice(THIN_LEVELS))]
    passes = [features_for(cell, 1.0, rng)] + [features_for(cell, level, rng) for level in levels]
    out = pd.concat(passes, ignore_index=True)
    out["root_id"] = int(root_id)
    out.to_parquet(path)
    return out

use_ids = qc.index[qc.use].to_numpy()
table = pd.concat(
    [
        df.groupby("keep_fraction", group_keys=False).apply(
            lambda g: g.sample(min(len(g), ROWS_PER_CELL), random_state=SEED)
        )
        for df in map(cell_features, use_ids)
    ],
    ignore_index=True,
)
META = ["is_axon", "vertex", "keep_fraction", "root_id"]
# The cache can hold columns from an earlier feature spec; train only on what the current spec makes.
current = set(features_for(load(use_ids[0]), 1.0, None).columns)
FEATURES = [c for c in table.columns if c not in META and c in current]
print(len(table), "rows from", table.root_id.nunique(), "cells;", len(FEATURES), "features;", f"{table.is_axon.mean():.0%} axon")

11700893 rows from 1969 cells; 52 features; 68% axon


## 4. Train

Cells are held out whole, so no cell contributes to both sides. Rows are weighted by cell type so rare types are not drowned out; the type sets the weight only and is not a feature.

In [6]:
train_idx, test_idx = next(
    GroupShuffleSplit(1, test_size=HOLDOUT_FRACTION, random_state=SEED).split(table, groups=table.root_id)
)
train, test = table.iloc[train_idx], table.iloc[test_idx]
test_ids = test.root_id.unique()

model = xgboost.XGBClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.5,   # no one feature is available to every tree
    min_child_weight=5,
    scale_pos_weight=float((~train.is_axon).sum() / max(train.is_axon.sum(), 1)),
    early_stopping_rounds=50,
    eval_metric="logloss",
    tree_method="hist",
    random_state=SEED,
)
# Rarer types weigh more, by the inverse square root of how common they are among training cells.
type_counts = pd.Series(type_of(train.root_id.unique())).value_counts()
type_weight = np.sqrt(type_counts.max() / type_counts)
model.fit(
    train[FEATURES], train.is_axon,
    sample_weight=type_weight.reindex(type_of(train.root_id)).to_numpy(),
    eval_set=[(test[FEATURES], test.is_axon)],
    verbose=False,
)
print("trees:", model.best_iteration + 1, " train cells:", train.root_id.nunique(), " held-out cells:", len(test_ids))

trees: 2000  train cells: 1575  held-out cells: 394


## 5. Evaluate

In [7]:
gain = pd.Series(model.get_booster().get_score(importance_type="total_gain"))
(gain / gain.sum()).sort_values(ascending=False).head(15).rename("share of total gain").to_frame().style.format("{:.1%}")

,share of total gain
down100um_syn_in_per_um,39.8%
down30um_syn_in_per_um,16.6%
down100um_syn_in_per_area,9.0%
down30um_syn_in_per_area,6.6%
down100um_syn_in,4.3%
down30um_syn_in,3.7%
down100um_syn_out,3.7%
down100um_syn_out_per_area,3.0%
down30um_cable_um,1.4%
down100um_cable_um,1.3%


In [8]:
test = test.assign(predicted=model.predict(test[FEATURES]).astype(bool))
test["correct"] = test.predicted == test.is_axon

def summarize(g):
    return pd.Series({
        "vertices": len(g),
        "accuracy": g.correct.mean(),
        "dendrite called axon": (g.predicted & ~g.is_axon).sum() / max((~g.is_axon).sum(), 1),
        "axon called dendrite": (~g.predicted & g.is_axon).sum() / max(g.is_axon.sum(), 1),
    })

print("Held-out vertices, by share of synapses kept (per-vertex, before any tree decode)")
test.groupby("keep_fraction").apply(summarize)

Held-out vertices, by share of synapses kept (per-vertex, before any tree decode)


,vertices,accuracy,dendrite called axon,axon called dendrite
keep_fraction,,,,
0.05,338317.0,0.978482,0.024099,0.020291
0.10,268869.0,0.979120,0.026910,0.018054
0.20,292785.0,0.985235,0.014447,0.014916
0.50,291315.0,0.988792,0.011064,0.011281
1.00,1157182.0,0.991745,0.006445,0.009113


In [9]:
per_cell = test[test.keep_fraction == 1].groupby("root_id").correct.mean()
print("Per-cell accuracy at full density: median", round(per_cell.median(), 3), " worst 5:")
per_cell.sort_values().head(5)

Per-cell accuracy at full density: median 0.993  worst 5:


root_id
864691135874914702    0.955667
864691135996891050    0.966333
864691135447820946    0.970000
864691135510267401    0.973431
864691135015048342    0.973464
Name: correct, dtype: float64

### Against the bundled model, on the same held-out cells

Both models label whole cells here, at full density and with 20% of synapses kept. Accuracy is weighted by cable length. `decoded` adds the tree decode (Viterbi on the skeleton, then absorbing small compartments) with the costs `s1_skeletons` uses.

In [10]:
old_encoder = ProbaVertexModel.from_config(SOURCE["bundled_model"])
new_encoder = ProbaVertexModel(model, FEATURES, feature_set="distance")
schema = TransitionSchema(
    classes=["dendrite", "axon"],
    transitions=COSTS,
    root_classes=["dendrite"],
)
decoder = StructuredLabeler(new_encoder, schema, absorb_min_size=ABSORB_MIN_SIZE)
old_decoder = StructuredLabeler(old_encoder, schema, absorb_min_size=ABSORB_MIN_SIZE)

def cable_accuracy(cell, predicted, truth):
    weight = np.asarray(cell.skeleton.half_edge_length, dtype=float)
    weight[cell.skeleton.root_positional] = 0
    return float((weight * (predicted == truth)).sum() / weight.sum())

rows = []
for root_id in test_ids[:50]:
    cell = load(root_id)
    truth = is_axon_of(cell)
    for keep in (1.0, 0.2):
        thinned = thin_synapses(cell, keep, np.random.default_rng([SEED, int(root_id) % 2**32, 1]))
        rows.append({
            "root_id": int(root_id),
            "synapses kept": keep,
            "bundled": cable_accuracy(cell, old_encoder.predict(thinned) == 1, truth),
            "bundled, decoded": cable_accuracy(cell, old_decoder.predict(thinned) == "axon", truth),
            "new": cable_accuracy(cell, new_encoder.predict(thinned) == 1, truth),
            "new, decoded": cable_accuracy(cell, decoder.predict(thinned) == "axon", truth),
        })
compare = pd.DataFrame(rows)
compare.groupby("synapses kept")[["bundled", "bundled, decoded", "new", "new, decoded"]].agg(["median", "min"]).round(3)

bundled        bundled, decoded           new         \
               median    min           median    min median    min   
synapses kept                                                        
0.2             0.939  0.826            0.966  0.866  0.990  0.973   
1.0             0.997  0.988            1.000  0.997  0.995  0.978   

              new, decoded         
                    median    min  
synapses kept                      
0.2                    1.0  0.990  
1.0                    1.0  0.977

### By cell type

Per-vertex accuracy on held-out cells, split by type, at full density and thinned. `train cells` is how many cells of the type the model saw.

In [11]:
test["cell_type"] = type_of(test.root_id)
test["density"] = np.where(test.keep_fraction == 1, "full", "thinned")
by_type = test.groupby(["cell_type", "density"]).apply(summarize)
by_type.insert(0, "cells", test.groupby(["cell_type", "density"]).root_id.nunique())
by_type.insert(1, "train cells", pd.Series(type_of(train.root_id.unique())).value_counts().reindex(by_type.index.get_level_values(0)).fillna(0).to_numpy())
by_type.drop(columns="vertices").style.format({"accuracy": "{:.1%}", "dendrite called axon": "{:.1%}", "axon called dendrite": "{:.1%}", "train cells": "{:.0f}"})

## 6. Transition costs

The decode above used costs set by hand (dendrite→axon 2, axon→dendrite 50, absorb 10). Here the encoder's unaries are computed once per cell and density, then decoded under every setting in a grid. The cells are held out from training and are not the ones compared above; every held-out cell of `TUNE_TYPE` is added so that type has as many cells as possible.

The labels come from synapse flow, which makes exactly one dendrite→axon transition per cell and no axon→dendrite one. So this sweep can show which costs hurt on clean cells, but it cannot show what a cost buys on a merge error, because there are none here.

In [12]:
from itertools import product

from ossify.structured_prediction import absorb_small_compartments, decode_tree

N_TUNE = 100
TUNE_TYPE = SOURCE["sparse_type"]      # every held-out cell of this type is included; None to skip
TUNE_KEEPS = (1.0, 0.2, 0.05, 0.02)
TO_AXON = [0, 1, 2, 5, 10, 20, 40]     # dendrite -> axon
TO_DENDRITE = [2, 5, 10, 20, 50, 100]  # axon -> dendrite
ABSORB = [0, 10, 30]                   # vertices
HAND_SET = (COSTS["dendrite", "axon"], COSTS["axon", "dendrite"], ABSORB_MIN_SIZE)                 # the setting used in section 5

tune_ids = np.union1d(test_ids[50 : 50 + N_TUNE], test_ids[type_of(test_ids) == TUNE_TYPE])

def sweep_cell(root_id):
    """Cable wrongly labelled, each way, under every cost setting, for one cell."""
    cell = load(root_id)
    skel = cell.skeleton
    truth = is_axon_of(cell)
    weight = np.asarray(skel.half_edge_length, dtype=float) / 1000
    weight[skel.root_positional] = 0
    rows = []
    for keep in TUNE_KEEPS:
        thinned = thin_synapses(cell, keep, np.random.default_rng([SEED, int(root_id) % 2**32, 2]))
        unaries = new_encoder.predict_unaries(thinned)
        for to_axon, to_dendrite in product(TO_AXON, TO_DENDRITE):
            costs = TransitionSchema(
                classes=["dendrite", "axon"],
                transitions={("axon", "dendrite"): to_dendrite, ("dendrite", "axon"): to_axon},
                root_classes=["dendrite"],
            )
            decoded, _ = decode_tree(skel, unaries, costs)
            for absorb in ABSORB:
                labels = absorb_small_compartments(skel, decoded, min_size=absorb) if absorb else decoded
                predicted = labels == 1
                rows.append({
                    "root_id": int(root_id), "keep": keep,
                    "to_axon": to_axon, "to_dendrite": to_dendrite, "absorb": absorb,
                    "cable_um": weight.sum(),
                    "dendrite called axon": weight[predicted & ~truth].sum() / weight.sum(),
                    "axon called dendrite": weight[~predicted & truth].sum() / weight.sum(),
                })
    return rows

# About 7 s per cell: the decode is a Python loop, so threads would not help.
sweep = pd.DataFrame([row for root_id in tune_ids for row in sweep_cell(root_id)])
sweep["error"] = sweep["dendrite called axon"] + sweep["axon called dendrite"]
sweep["cell_type"] = type_of(sweep.root_id)
print(sweep.root_id.nunique(), "cells,", len(TUNE_KEEPS), "densities,", len(TO_AXON) * len(TO_DENDRITE) * len(ABSORB), "settings")

104 cells, 4 densities, 126 settings


In [13]:
SETTING = ["to_axon", "to_dendrite", "absorb"]
by_setting = sweep.groupby(SETTING).error.agg(mean="mean", worst="max")
best = by_setting["mean"].idxmin()
print("lowest mean error (to_axon, to_dendrite, absorb):", best, by_setting.loc[best].round(4).to_dict())
print("hand-set:", HAND_SET, by_setting.loc[HAND_SET].round(4).to_dict())
by_setting.sort_values("mean").head(10).style.format("{:.2%}")

lowest mean error (to_axon, to_dendrite, absorb): (np.int64(40), np.int64(100), np.int64(30)) {'mean': 0.0009, 'worst': 0.0441}
hand-set: (2, 50, 10) {'mean': 0.0026, 'worst': 0.0505}


In [14]:
# Mean error over cells for each pair of costs, by density, at the best absorb size.
(
    sweep[sweep.absorb == best[2]]
    .pivot_table(index="to_axon", columns=["keep", "to_dendrite"], values="error")
    .style.format("{:.2%}").background_gradient(axis=None, cmap="magma_r")
)

Error by cell type, at the best setting and at the hand-set one.

In [15]:
def at(setting):
    return sweep[(sweep[SETTING] == setting).all(axis=1)]

pd.concat({"best": at(best), "hand-set": at(HAND_SET)}, names=["setting"]).groupby(
    ["cell_type", "keep", "setting"]
)[["error", "dendrite called axon", "axon called dendrite"]].mean().unstack("setting").style.format("{:.2%}")

## 7. Save

Written into `src/ossify/compartment_models`, in the same layout as the bundled models, so it loads with `load_labeler("<dataset>_distance")`.

In [16]:
from ossify.compartments import MODEL_DIR

# Saved beside the bundled models, in the same layout, so it loads by name.
name = f"{DATASET}_distance"
model.save_model(MODEL_DIR / "xgb" / f"{name}.ubj")
(MODEL_DIR / f"{name}.json").write_text(json.dumps({
    "dataset": DATASET,
    "description": "Axon/dendrite on proofread cells. Downstream windows of 10, 30 and 100 um; robust to missing synapses.",
    "downstream_hops": 0,
    "upstream_hops": 0,
    "bidirectional_hops": 0,
    "feature_set": "distance",
    "feature_columns": FEATURES,
    "model_file": f"xgb/{name}.ubj",
    "decode": {"schema": schema.to_dict(), "absorb_min_size": ABSORB_MIN_SIZE},
    "training": {
        "datastack": SOURCE["datastack"],
        "version": SOURCE["version"],
        "cells": SOURCE["cells"],
        "qc": QC,
        "n_train_cells": int(train.root_id.nunique()),
        "thin_levels": THIN_LEVELS,
    },
}, indent=2) + "\n")
print("saved; load with load_labeler(" + repr(name) + ")")


saved; load with ProbaVertexModel.from_config('minnie65_distance')
